# Consolidate Collection Lineage CSVs and Cross-Check with EOSDIS metadataAssociations

This notebook combines the per-collection lineage CSVs produced by
**Collection_Lineage_via_ATBD_Skill** into one file per collection folder, then checks
every association against the database snapshot in `eosdis_collections_w_metadata_associations.json`.

**Input**
- `Reports/<DAAC>/<FOLDER>/*_lineage.csv` for `<DAAC>` in `ASF`, `GES_DISC`, `LARC_CLOUD`
- `eosdis_collections_w_metadata_associations.json` (snapshot from `graphql.earthdata.nasa.gov`)

**Output**
- `Reports/<DAAC>/<FOLDER>_lineage.csv`: all rows from the folder's CSVs plus a `Match` column.
  For example, `Reports/ASF/OPERA/*_lineage.csv` becomes `Reports/ASF/OPERA_lineage.csv`.

**`Match` values**

| Value | Meaning |
|---|---|
| `true` | The related collection is in the DB record's `metadataAssociations` with the **same** type |
| `<TYPE>` (e.g. `INPUT`) | The related collection is in the DB record but with a **different** type. The DB type is shown |
| `false` | The related collection is not in the DB record's `metadataAssociations`, or the source collection is not in the snapshot |

In [1]:
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

# ---- Inputs ---------------------------------------------------------------
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
REPORTS_DIR = REPO_ROOT / "Reports"
DAACS = ["ASF", "GES_DISC", "LARC_CLOUD"]
DB_SNAPSHOT = REPO_ROOT / "eosdis_collections_w_metadata_associations.json"

# ---- Options --------------------------------------------------------------
DRY_RUN = False   # True = build and summarize without writing any CSVs

## Step 1: Load the database snapshot

Index every collection by `(shortName, version)`, and separately by `shortName` alone.
Some short names (e.g. `GloSSAC`) have several versions in the snapshot, so the version is
used to pick the right record.

In [2]:
def norm(s):
    # Trim and uppercase for comparison. Some DB entryIds have trailing spaces,
    # e.g. "OPERA_L2_CSLC-S1-STATIC_V1 "
    return (s or "").strip().upper()

snapshot = json.loads(DB_SNAPSHOT.read_text())
db_by_key, db_by_name = {}, defaultdict(list)
for c in snapshot["collections"]:
    db_by_key[(norm(c["shortName"]), (c.get("version") or "").strip())] = c
    db_by_name[norm(c["shortName"])].append(c)

print(f"Snapshot source   : {snapshot['source']}")
print(f"Retrieved (UTC)   : {snapshot['retrieved_at_utc']}")
print(f"Collections       : {len(snapshot['collections'])}")
print(f"By data center    : {dict(Counter(c['dataCenter'] for c in snapshot['collections']))}")

Snapshot source   : https://graphql.earthdata.nasa.gov/api
Retrieved (UTC)   : 2026-11-17T16:03:08.655405+00:00
Collections       : 750
By data center    : {'ASF': 41, 'GES_DISC': 157, 'GHRC_DAAC': 1, 'LANCEMODIS': 20, 'LARC_CLOUD': 526, 'LPCLOUD': 3, 'LPDAAC_ECS': 2}


## Step 2: Define the matching rules

1. **Find the source collection.** Look up the CSV `shortname` + `version` in the snapshot.
   If the exact version isn't there and only one record has that short name, use that record.
2. **Find the related collection.** Compare the CSV `related_shortname` with each
   `metadataAssociations[].entryId` of that record. The comparison ignores case and surrounding spaces.
   The association version is not compared, because the DB often leaves it `null`.
3. **Compare the type.** Same `association_type` gives `true`. A different type gives the DB type
   (if the DB lists the same entry under more than one type, they are joined with `|`).
   No matching entry gives `false`.

In [3]:
def find_db_collection(shortname, version):
    name = norm(shortname)
    rec = db_by_key.get((name, (version or "").strip()))
    if rec is None and len(db_by_name.get(name, [])) == 1:
        rec = db_by_name[name][0]
    return rec


def match_value(rec, row):
    if rec is None:
        return "false"
    related = norm(row["related_shortname"])
    db_types = [a.get("type", "").strip() for a in rec.get("metadataAssociations") or []
                if norm(a.get("entryId")) == related]
    if not db_types:
        return "false"
    if norm(row["association_type"]) in {norm(t) for t in db_types}:
        return "true"
    return "|".join(dict.fromkeys(db_types))


# Sanity check with the example from the request: OPERA_L2_CSLC-S1_V1
rec = find_db_collection("OPERA_L2_CSLC-S1_V1", "1")
for rel, typ in [("SENTINEL-1A_SLC", "PARENT"), ("NISAR_DEM", "INPUT")]:
    print(f"{rel:<18} csv={typ:<7} -> Match={match_value(rec, {'related_shortname': rel, 'association_type': typ})}")

SENTINEL-1A_SLC    csv=PARENT  -> Match=INPUT
NISAR_DEM          csv=INPUT   -> Match=false


## Step 3: Discover the collection folders

Each subfolder of `Reports/<DAAC>/` is one collection suite. Only per-collection `*_lineage.csv` files are
combined. Two kinds of file are skipped:
- `*_lineage_evidence.json`
- `*_suite_lineage.csv`: a roll-up of the per-collection CSVs in the same folder. Including it would
  double every row. The code below confirms each suite file has exactly the same rows.

In [4]:
folders = []
for daac in DAACS:
    for folder in sorted(p for p in (REPORTS_DIR / daac).iterdir() if p.is_dir()):
        suite = sorted(folder.glob("*_suite_lineage.csv"))
        csvs = sorted(f for f in folder.glob("*_lineage.csv") if f not in suite)
        if csvs:
            folders.append((daac, folder, csvs, suite))

def read_rows(files):
    return sorted(tuple(r.values()) for f in files for r in csv.DictReader(f.open(newline="")))

print(f"{'DAAC':<11} {'Folder':<15} {'CSV files':>9}  suite file")
for daac, folder, csvs, suite in folders:
    check = ("(same rows as per-collection CSVs)" if read_rows(suite) == read_rows(csvs)
             else "(DIFFERS from per-collection CSVs!)") if suite else "-"
    print(f"{daac:<11} {folder.name:<15} {len(csvs):>9}  {' '.join(f.name for f in suite)} {check}")
print(f"\n{len(folders)} folders, {sum(len(f[2]) for f in folders)} per-collection CSV files")

DAAC        Folder          CSV files  suite file
ASF         ARIA                    1   -
ASF         OPERA                   8   -
ASF         SENTINEL-1              2   -
ASF         SENTINEL-1C            14   -
ASF         SENTINEL-1D            14   -
ASF         UAVSAR                  2   -
GES_DISC    AIRS                   39  AIRS_suite_lineage.csv (same rows as per-collection CSVs)
GES_DISC    CAR                    16  CAR_BRDF_suite_lineage.csv (same rows as per-collection CSVs)
GES_DISC    LHSAS                   3  LHASA_suite_lineage.csv (same rows as per-collection CSVs)
GES_DISC    MEaSUREs_OMI            4  OMI_MEaSUREs_suite_lineage.csv (same rows as per-collection CSVs)
GES_DISC    MINDS                   1   -
GES_DISC    TOVS                    9  TOVS_PathA_V02_suite_lineage.csv (same rows as per-collection CSVs)
LARC_CLOUD  CALIPSO                35  CALIPSO_suite_lineage.csv (same rows as per-collection CSVs)
LARC_CLOUD  CATS                   16  CATS_suit

## Step 4: Consolidate, add `Match`, and write `<FOLDER>_lineage.csv`

Rows keep the original column order and are concatenated in file-name order. `Match` is added as
the last column. Each output file goes in the DAAC directory, next to its source folder.

In [5]:
results = []      # one entry per folder
all_rows = []     # every consolidated row, tagged with DAAC/folder

for daac, folder, csvs, _ in folders:
    rows, fieldnames = [], None
    for f in csvs:
        with f.open(newline="") as fh:
            reader = csv.DictReader(fh)
            fieldnames = fieldnames or reader.fieldnames
            for row in reader:
                rec = find_db_collection(row["shortname"], row["version"])
                row["Match"] = match_value(rec, row)
                row["_in_db"] = rec is not None
                rows.append(row)

    out_path = folder.parent / f"{folder.name}_lineage.csv"
    if not DRY_RUN:
        with out_path.open("w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=fieldnames + ["Match"], extrasaction="ignore", lineterminator="\n")
            w.writeheader()
            w.writerows(rows)

    counts = Counter("true" if r["Match"] == "true" else "false" if r["Match"] == "false" else "type_differs" for r in rows)
    results.append({"daac": daac, "folder": folder.name, "out": out_path, "files": len(csvs), "rows": len(rows), **counts})
    all_rows += [{**r, "_daac": daac, "_folder": folder.name} for r in rows]

action = "Would write" if DRY_RUN else "Wrote"
print(f"{'Output file':<44} {'files':>5} {'rows':>5} {'true':>5} {'type≠':>5} {'false':>5}")
for r in results:
    print(f"{str(r['out'].relative_to(REPO_ROOT)):<44} {r['files']:>5} {r['rows']:>5} "
          f"{r.get('true', 0):>5} {r.get('type_differs', 0):>5} {r.get('false', 0):>5}")
print(f"\n{action} {len(results)} consolidated files, {len(all_rows)} rows total")

Output file                                  files  rows  true type≠ false
Reports/ASF/ARIA_lineage.csv                     1     6     0     1     5
Reports/ASF/OPERA_lineage.csv                    8    40     6    17    17
Reports/ASF/SENTINEL-1_lineage.csv               2     8     0     3     5
Reports/ASF/SENTINEL-1C_lineage.csv             14   100    28    13    59
Reports/ASF/SENTINEL-1D_lineage.csv             14    98    42    14    42
Reports/ASF/UAVSAR_lineage.csv                   2    15     2     0    13
Reports/GES_DISC/AIRS_lineage.csv               39   196     0     2   194
Reports/GES_DISC/CAR_lineage.csv                16    21     0     0    21
Reports/GES_DISC/LHSAS_lineage.csv               3     5     0     0     5
Reports/GES_DISC/MEaSUREs_OMI_lineage.csv        4    18     0     0    18
Reports/GES_DISC/MINDS_lineage.csv               1     5     0     0     5
Reports/GES_DISC/TOVS_lineage.csv                9    75     0     1    74
Reports/LARC_CLOUD/CALIPS

## Step 5: Summary

### 5a. Match results by DAAC

In [6]:
print(f"{'DAAC':<11} {'rows':>5} {'true':>5} {'type≠':>6} {'false':>6} {'% true':>7}")
for daac in DAACS + ["ALL"]:
    rs = [r for r in all_rows if daac in ("ALL", r["_daac"])]
    t = sum(r["Match"] == "true" for r in rs)
    f = sum(r["Match"] == "false" for r in rs)
    d = len(rs) - t - f
    print(f"{daac:<11} {len(rs):>5} {t:>5} {d:>6} {f:>6} {100 * t / len(rs):>6.1f}%")

DAAC         rows  true  type≠  false  % true
ASF           267    78     48    141   29.2%
GES_DISC      320     0      3    317    0.0%
LARC_CLOUD   1037    33     80    924    3.2%
ALL          1624   111    131   1382    6.8%


### 5b. Type differences: CSV `association_type` vs. DB type

Associations where both sources name the same related collection but disagree on the type.

In [7]:
diff = Counter((r["association_type"], r["Match"]) for r in all_rows if r["Match"] not in ("true", "false"))
print(f"{'CSV type':<20} {'DB type':<22} {'count':>5}")
for (csv_t, db_t), n in diff.most_common():
    print(f"{csv_t:<20} {db_t:<22} {n:>5}")

CSV type             DB type                count
CHILD                DEPENDENT                 32
PARENT               INPUT                     26
RELATED              PARENT                    19
PARENT               DEPENDENT                 14
CHILD                SCIENCE ASSOCIATED        13
RELATED              SCIENCE ASSOCIATED        12
RELATED              CHILD                     11
PARENT               SCIENCE ASSOCIATED         3
PARENT               RELATED                    1


### 5c. Source collections not in the database snapshot

Every row for these collections is `false`, because there is no DB record to compare with.

In [8]:
missing = sorted({(r["_daac"], r["_folder"], r["shortname"], r["version"]) for r in all_rows if not r["_in_db"]})
print(f"{len(missing)} source collections not found in the snapshot")
for m in missing:
    print("  ", " / ".join(m))

0 source collections not found in the snapshot


### 5d. DB associations the CSVs don't list

For each source collection in the CSVs, these are the DB `metadataAssociations` entries that no CSV
row names. The `Match` column can't show them because it only covers rows that are in the CSVs.

In [9]:
csv_related = defaultdict(set)
csv_src = {}
for r in all_rows:
    key = (norm(r["shortname"]), r["version"].strip())
    csv_related[key].add(norm(r["related_shortname"]))
    csv_src[key] = (r["_daac"], r["_folder"], r["shortname"], r["version"])

db_only = []
for key, related in csv_related.items():
    rec = find_db_collection(*key)
    for a in (rec or {}).get("metadataAssociations") or []:
        if norm(a.get("entryId")) not in related:
            db_only.append((*csv_src[key], a.get("type"), (a.get("entryId") or "").strip(), a.get("version")))

self_refs = sorted(d for d in db_only if norm(d[5]) == norm(d[2]))
other_refs = sorted(d for d in db_only if norm(d[5]) != norm(d[2]))
print(f"{len(db_only)} DB associations not in the CSVs")
print(f"  {len(self_refs):>4} point back to the collection itself (5d-i)")
print(f"  {len(other_refs):>4} point to a different collection (5d-ii)\n")
print(f"{'DAAC':<11} {'self':>5} {'other':>6}")
for daac in DAACS:
    print(f"{daac:<11} {sum(d[0] == daac for d in self_refs):>5} {sum(d[0] == daac for d in other_refs):>6}")


def print_assocs(items):
    print(f"{'DAAC / Folder':<26} {'Collection':<56} {'ver':<11} {'DB type':<19} {'DB entryId':<60} {'DB ver'}")
    for daac, folder, name, ver, typ, entry, entry_ver in items:
        print(f"{daac + ' / ' + folder:<26} {name:<56} {ver:<11} {str(typ):<19} {entry:<60} {entry_ver or '-'}")

544 DB associations not in the CSVs
   224 point back to the collection itself (5d-i)
   320 point to a different collection (5d-ii)

DAAC         self  other
ASF             0      9
GES_DISC       68      1
LARC_CLOUD    156    310


#### 5d-i. Associations that point back to the collection itself

The DB `entryId` is the collection's own short name. Usually this links to another version of the same
collection (e.g. ERBE Edition2 to Edition3), or it's a placeholder `SCIENCE ASSOCIATED` entry
(most GES_DISC records).

In [10]:
print(f"{len(self_refs)} self-referencing associations\n")
print_assocs(self_refs)

224 self-referencing associations

DAAC / Folder              Collection                                               ver         DB type             DB entryId                                                   DB ver
GES_DISC / AIRS            AIRABRAD                                                 005         SCIENCE ASSOCIATED  AIRABRAD                                                     005
GES_DISC / AIRS            AIRG2SSD_IRonly                                          006         SCIENCE ASSOCIATED  AIRG2SSD_IRonly                                              006
GES_DISC / AIRS            AIRG2SSD_IRonly                                          7.0         SCIENCE ASSOCIATED  AIRG2SSD_IRonly                                              7.0
GES_DISC / AIRS            AIRH2CCF                                                 7.0         SCIENCE ASSOCIATED  AIRH2CCF                                                     7.0
GES_DISC / AIRS            AIRH2RET                      

#### 5d-ii. Associations that point to a different collection

These are real lineage links in the DB that the ATBD-derived CSVs missed. They're candidates to review
or add to the lineage output.

In [11]:
print(f"{len(other_refs)} associations to a different collection\n")
print_assocs(other_refs)

320 associations to a different collection

DAAC / Folder              Collection                                               ver         DB type             DB entryId                                                   DB ver
ASF / OPERA                OPERA_L2_CSLC-S1-STATIC_V1                               1           INPUT               SENTINEL-1D_SLC                                              -
ASF / OPERA                OPERA_L2_RTC-S1-STATIC_V1                                1           INPUT               SENTINEL-1D_SLC                                              -
ASF / OPERA                OPERA_L3_DIST-ALERT-S1_V1                                1           SCIENCE ASSOCIATED  SENTINEL-1A_SLC                                              1
ASF / OPERA                OPERA_L3_DIST-ALERT-S1_V1                                1           SCIENCE ASSOCIATED  SENTINEL-1C_SLC                                              1
ASF / OPERA                OPERA_L3_DIST-ALERT-S1_V1    

### 5e. Sample output: `Reports/ASF/OPERA_lineage.csv`

In [12]:
sample = next(r["out"] for r in results if r["folder"] == "OPERA")
print(sample.read_text() if sample.exists() else "(not written: DRY_RUN=True)")

shortname,version,association_type,related_shortname,related_version,confidence,lineage_relationship,Match
OPERA_L2_CSLC-S1-STATIC_V1,1,INPUT,NISAR_DEM,1,HIGH,input_to,false
OPERA_L2_CSLC-S1-STATIC_V1,1,INPUT,SENTINEL-1A_SLC,1,MEDIUM,input_to,true
OPERA_L2_CSLC-S1-STATIC_V1,1,INPUT,SENTINEL-1B_SLC,1,MEDIUM,input_to,true
OPERA_L2_CSLC-S1-STATIC_V1,1,INPUT,SENTINEL-1C_SLC,1,MEDIUM,input_to,true
OPERA_L2_CSLC-S1-STATIC_V1,1,CHILD,OPERA_L3_DISP-S1-STATIC_V1,1,HIGH,downstream_product,false
OPERA_L2_CSLC-S1-STATIC_V1,1,CHILD,OPERA_L3_DISP-S1_V1,1,HIGH,downstream_product,false
OPERA_L2_CSLC-S1-STATIC_V1,1,RELATED,OPERA_L2_CSLC-S1_V1,1,HIGH,parallel_product,SCIENCE ASSOCIATED
OPERA_L2_CSLC-S1_V1,1,PARENT,SENTINEL-1A_SLC,1,MEDIUM,derived_from,INPUT
OPERA_L2_CSLC-S1_V1,1,PARENT,SENTINEL-1B_SLC,1,MEDIUM,derived_from,INPUT
OPERA_L2_CSLC-S1_V1,1,PARENT,SENTINEL-1C_SLC,1,MEDIUM,derived_from,INPUT
OPERA_L2_CSLC-S1_V1,1,PARENT,SENTINEL-1D_SLC,1,MEDIUM,derived_from,INPUT
OPERA_L2_CSLC-S1_V1,1,INPUT,GNS

## Summary

**What was done**
1. Combined 388 per-collection `*_lineage.csv` files from 30 collection folders under
   `Reports/ASF`, `Reports/GES_DISC` and `Reports/LARC_CLOUD` into 30 files named `Reports/<DAAC>/<FOLDER>_lineage.csv`.
   That's 1,624 rows. The 24 `*_suite_lineage.csv` roll-ups were left out because each one repeats its folder's per-collection rows exactly.
2. Added a `Match` column by checking each row against `eosdis_collections_w_metadata_associations.json`
   (750 collections, retrieved 2026-11-17).

**Results** (1,624 rows)

| DAAC | rows | `true` | type differs | `false` | % `true` |
|---|---:|---:|---:|---:|---:|
| ASF | 267 | 78 | 48 | 141 | 29.2% |
| GES_DISC | 320 | 0 | 3 | 317 | 0.0% |
| LARC_CLOUD | 1,037 | 33 | 80 | 924 | 3.2% |
| **All** | **1,624** | **111** | **131** | **1,382** | **6.8%** |

**Observations**
- **The CSVs and the DB mostly describe different associations.** 85% of CSV rows have no matching DB
  entry. Going the other way, 544 DB associations are missing from the CSVs. 224 of those point back to the
  collection itself, for example every GES_DISC AIRS/CAR/TOVS record has only a `SCIENCE ASSOCIATED`
  entry pointing to its own short name. That's why GES_DISC has 0% `true`.
- **Type disagreements follow a pattern.** The CSVs use `CHILD`/`PARENT` where the DB uses `DEPENDENT`/`INPUT`
  (CHILD→DEPENDENT 32, PARENT→INPUT 26, PARENT→DEPENDENT 14). The CSVs use `RELATED` where the DB has a
  directional type (PARENT 19, CHILD 11) or `SCIENCE ASSOCIATED` (12).
- ASF agrees with the DB most often (29% `true`), mainly the SENTINEL-1C/1D suites.
- **Data quality problems in the snapshot or the CSVs** (these show up as `false` or as DB-only entries):
  - CSV typo: `OPERA_L4_TROPO-ZENITH_V1` lists `ASF_ECMWF_TROP`, but the DB has `ASF_ECMWF_TROPO`.
  - DB typos: `SENTINEL-1_BURSTS` → `SENIINEL-1C_SLC` / `SENIINEL-1D_SLC`.
  - The DB has a free-text entryId: `OPeNDAP data access for CAL_IIR_L2_Track-Standard-V4-20`.
  - The DB has a trailing space in an entryId (`"OPERA_L2_CSLC-S1-STATIC_V1 "`). The matching here trims spaces.
  - Some LHASA DB associations have no `type`.
- `Reports/LARC_CLOUD/SCAR_A_lineage.csv` has only a header row, because both source CSVs were empty.

**How to re-run:** set `DRY_RUN = True` in the first code cell to preview the results without writing files.